# NPPE-2 Best-of-Both Safe Notebook

This notebook is built to be **stable first, competitive second**.

What it does:
- keeps the strong **`whisper-small + beam=5`** inference path from the notebook that scored around **0.8**
- adds a **crash-proof full fine-tuning** path for `whisper-small` on your train split
- evaluates **baseline vs fine-tuned vs hybrid fallback** on validation
- chooses the best strategy before creating `submission.csv`

Why no LoRA here by default:
- your recent crash is the known Whisper + generic PEFT wrapper clash around `input_ids` vs `input_features`
- this notebook avoids that trap completely in the main path

You can still submit the zero-shot baseline even if you skip training.


In [ ]:
!pip install -q -U transformers datasets accelerate evaluate jiwer soundfile librosa scikit-learn


In [ ]:
import os
import re
import gc
import math
import random
import string
import unicodedata
from dataclasses import dataclass
from typing import Any, Dict, List, Union

import numpy as np
import pandas as pd
import torch
import soundfile as sf
import librosa
import evaluate
import matplotlib.pyplot as plt

from tqdm.auto import tqdm
from sklearn.model_selection import train_test_split

from datasets import Dataset, Audio
from jiwer import wer as jiwer_wer, cer as jiwer_cer

from transformers import (
    WhisperProcessor,
    WhisperForConditionalGeneration,
    Seq2SeqTrainer,
    Seq2SeqTrainingArguments,
    EarlyStoppingCallback,
    set_seed,
)

SEED = 42
set_seed(SEED)
random.seed(SEED)
np.random.seed(SEED)

DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
DTYPE = torch.float16 if DEVICE == "cuda" else torch.float32

# ---- Config ----
MODEL_NAME = "openai/whisper-small"
VAL_SIZE = 0.12
RUN_BASELINE_VAL = True
DO_TRAIN = True
USE_BASELINE_FALLBACK = True

BEST_MODEL_METRIC = "cleaned_wer"   # switch to "raw_wer" if leaderboard seems stricter

PER_DEVICE_TRAIN_BATCH_SIZE = 4
PER_DEVICE_EVAL_BATCH_SIZE = 8
GRAD_ACCUM = 4
NUM_EPOCHS = 3
LEARNING_RATE = 1e-5
WEIGHT_DECAY = 0.01
WARMUP_RATIO = 0.05
NUM_BEAMS = 5
MAX_NEW_TOKENS = 255

OUTPUT_DIR = "/kaggle/working/whisper_small_fullft"
FINAL_DIR = "/kaggle/working/whisper_small_best"

CANDIDATE_INPUTS = [
    "/kaggle/input/nppe-2-multilingual-speech-recognition",
    "/kaggle/input/competitions/multilingual-speech-recognition",
    "/kaggle/input/multilingual-speech-recognition",
]

def detect_input_root():
    for root in CANDIDATE_INPUTS:
        if os.path.exists(root):
            return root
    raise FileNotFoundError(f"Could not find competition data. Checked: {CANDIDATE_INPUTS}")

INPUT_ROOT = detect_input_root()
TRAIN_CSV = os.path.join(INPUT_ROOT, "train.csv")
TEST_CSV = os.path.join(INPUT_ROOT, "test.csv")
SAMPLE_SUB_CSV = os.path.join(INPUT_ROOT, "sample_submission.csv")

print("DEVICE:", DEVICE)
print("INPUT_ROOT:", INPUT_ROOT)


In [ ]:
def normalize_raw_text(x: str) -> str:
    return re.sub(r"\s+", " ", str(x).strip())

def submission_cleaner(text: str) -> str:
    text = str(text).lower().strip()
    text = re.sub(r"[^\w\s\u0900-\u097F\u0B80-\u0BFF]", "", text)
    text = re.sub(r"\s+", " ", text).strip()
    return text

def detect_script(text: str) -> str:
    counts = {}
    for ch in str(text):
        if not ch.isalpha():
            continue
        name = unicodedata.name(ch, "")
        for script in [
            "LATIN", "TAMIL", "DEVANAGARI", "BENGALI", "TELUGU",
            "KANNADA", "MALAYALAM", "GUJARATI", "GURMUKHI",
            "ORIYA", "ARABIC", "CYRILLIC"
        ]:
            if script in name:
                counts[script] = counts.get(script, 0) + 1
                break
        else:
            counts["OTHER"] = counts.get("OTHER", 0) + 1
    return max(counts, key=counts.get) if counts else "OTHER"

def resolve_audio_path(root: str, audio_name: str) -> str:
    candidates = [
        os.path.join(root, "competition_data", audio_name),
        os.path.join(root, "competition_data", "train", audio_name),
        os.path.join(root, "competition_data", "test", audio_name),
        os.path.join(root, audio_name),
    ]
    for p in candidates:
        if os.path.exists(p):
            return p
    raise FileNotFoundError(f"Audio file not found for {audio_name}. Tried: {candidates}")

def get_duration(path: str) -> float:
    info = sf.info(path)
    return info.frames / info.samplerate

def compute_text_metrics(preds, refs):
    raw_preds = [normalize_raw_text(x) for x in preds]
    raw_refs = [normalize_raw_text(x) for x in refs]
    clean_preds = [submission_cleaner(x) for x in preds]
    clean_refs = [submission_cleaner(x) for x in refs]

    ref_words = np.array([max(1, len(x.split())) for x in clean_refs], dtype=np.float32)
    pred_words = np.array([len(x.split()) for x in clean_preds], dtype=np.float32)

    return {
        "raw_wer": jiwer_wer(raw_refs, raw_preds),
        "raw_cer": jiwer_cer(raw_refs, raw_preds),
        "cleaned_wer": jiwer_wer(clean_refs, clean_preds),
        "cleaned_cer": jiwer_cer(clean_refs, clean_preds),
        "exact_match_clean": float(np.mean([p == r for p, r in zip(clean_preds, clean_refs)])),
        "empty_pred_rate": float(np.mean([p == "" for p in clean_preds])),
        "pred_ref_word_ratio_clean": float(np.mean(pred_words / ref_words)),
    }

def is_risky_prediction(text: str) -> bool:
    toks = submission_cleaner(text).split()
    if len(toks) == 0:
        return True
    for i in range(len(toks) - 2):
        if toks[i] == toks[i + 1] == toks[i + 2]:
            return True
    if len(toks) >= 6:
        bigrams = [" ".join(toks[i:i+2]) for i in range(len(toks) - 1)]
        counts = {}
        for bg in bigrams:
            counts[bg] = counts.get(bg, 0) + 1
        if max(counts.values()) >= 3:
            return True
    if len(toks) >= 8 and len(set(toks)) <= max(2, len(toks) // 4):
        return True
    return False

def build_hybrid_predictions(fine_preds, base_preds):
    hybrid = []
    used_fallback = 0
    for fp, bp in zip(fine_preds, base_preds):
        if is_risky_prediction(fp):
            hybrid.append(bp)
            used_fallback += 1
        else:
            hybrid.append(fp)
    return hybrid, used_fallback


In [ ]:
train_df = pd.read_csv(TRAIN_CSV)
test_df = pd.read_csv(TEST_CSV)
sample_sub = pd.read_csv(SAMPLE_SUB_CSV)

audio_col = "audio" if "audio" in train_df.columns else train_df.columns[0]
text_col = "text" if "text" in train_df.columns else train_df.columns[-1]

train_df[text_col] = train_df[text_col].map(normalize_raw_text)
train_df["audio_path"] = train_df[audio_col].map(lambda x: resolve_audio_path(INPUT_ROOT, x))
test_df["audio_path"] = test_df[audio_col].map(lambda x: resolve_audio_path(INPUT_ROOT, x))

train_df["duration"] = train_df["audio_path"].map(get_duration)
train_df["n_words"] = train_df[text_col].map(lambda x: len(str(x).split()))
train_df["n_chars"] = train_df[text_col].map(lambda x: len(str(x)))
train_df["script"] = train_df[text_col].map(detect_script)
train_df["chars_per_sec"] = train_df["n_chars"] / train_df["duration"].clip(lower=1e-6)
train_df["words_per_sec"] = train_df["n_words"] / train_df["duration"].clip(lower=1e-6)

script_counts = train_df["script"].value_counts()
train_df["script_bucket"] = train_df["script"].where(train_df["script"].map(script_counts) >= 5, "RARE")

print("Total train samples:", len(train_df))
print("Total test samples:", len(test_df))
print("Total hours:", round(train_df["duration"].sum() / 3600, 3))
print("Average duration:", round(train_df["duration"].mean(), 3))
print("Max duration:", round(train_df["duration"].max(), 3))
print("Samples with <3 words:", int((train_df["n_words"] < 3).sum()))
print("\nScript distribution:")
print(train_df["script_bucket"].value_counts())

display(train_df[[audio_col, text_col, "duration", "n_words", "script_bucket"]].head())

fig, axes = plt.subplots(1, 3, figsize=(18, 4))
train_df["duration"].hist(bins=30, ax=axes[0])
axes[0].set_title("Duration")
train_df["n_words"].hist(bins=30, ax=axes[1])
axes[1].set_title("Words per transcript")
axes[2].scatter(train_df["duration"], train_df["n_words"], s=10)
axes[2].set_title("Duration vs words")
axes[2].set_xlabel("duration")
axes[2].set_ylabel("n_words")
plt.show()


In [ ]:
train_df["dur_bin"] = pd.qcut(train_df["duration"], q=4, duplicates="drop").astype(str)
train_df["stratify_key"] = train_df["script_bucket"] + "__" + train_df["dur_bin"]

combo_counts = train_df["stratify_key"].value_counts()
train_df.loc[train_df["stratify_key"].map(combo_counts) < 2, "stratify_key"] = train_df["script_bucket"]

try:
    tr_df, va_df = train_test_split(
        train_df,
        test_size=VAL_SIZE,
        random_state=SEED,
        stratify=train_df["stratify_key"],
    )
except ValueError:
    tr_df, va_df = train_test_split(
        train_df,
        test_size=VAL_SIZE,
        random_state=SEED,
        stratify=train_df["script_bucket"],
    )

tr_df = tr_df.reset_index(drop=True)
va_df = va_df.reset_index(drop=True)

print("Train size:", len(tr_df))
print("Val size:", len(va_df))
print("\nTrain script mix:")
print(tr_df["script_bucket"].value_counts(normalize=True).round(3))
print("\nVal script mix:")
print(va_df["script_bucket"].value_counts(normalize=True).round(3))


In [ ]:
def load_infer_bundle(model_name=MODEL_NAME):
    processor = WhisperProcessor.from_pretrained(model_name, task="transcribe")
    model = WhisperForConditionalGeneration.from_pretrained(
        model_name,
        torch_dtype=DTYPE,
        low_cpu_mem_usage=True,
    )
    model.to(DEVICE)
    model.eval()
    model.config.forced_decoder_ids = None
    model.generation_config.forced_decoder_ids = None
    model.generation_config.task = "transcribe"
    return processor, model

@torch.no_grad()
def transcribe_paths(audio_paths, processor, model, batch_size=8, desc="Transcribing"):
    preds = []
    for start in tqdm(range(0, len(audio_paths), batch_size), desc=desc):
        batch_paths = audio_paths[start:start+batch_size]
        features = []
        for p in batch_paths:
            speech, _ = librosa.load(p, sr=16000)
            feat = processor(speech, sampling_rate=16000, return_tensors="pt").input_features[0]
            features.append(feat)
        input_features = torch.stack(features)
        input_features = input_features.to(DEVICE, dtype=DTYPE if DEVICE == "cuda" else torch.float32)
        generated = model.generate(
            input_features=input_features,
            num_beams=NUM_BEAMS,
            max_new_tokens=MAX_NEW_TOKENS,
            task="transcribe",
        )
        texts = processor.batch_decode(generated, skip_special_tokens=True)
        preds.extend([submission_cleaner(x) for x in texts])
    return preds

baseline_processor, baseline_model = load_infer_bundle()

baseline_val_preds = None
baseline_val_metrics = None

if RUN_BASELINE_VAL:
    baseline_val_preds = transcribe_paths(
        va_df["audio_path"].tolist(),
        baseline_processor,
        baseline_model,
        batch_size=PER_DEVICE_EVAL_BATCH_SIZE,
        desc="Baseline val"
    )
    baseline_val_metrics = compute_text_metrics(baseline_val_preds, va_df[text_col].tolist())
    print("Baseline val metrics:")
    print(pd.Series(baseline_val_metrics))


In [ ]:
def to_hf_dataset(df: pd.DataFrame, include_text: bool = True) -> Dataset:
    data = {"audio": df["audio_path"].tolist()}
    if include_text:
        data["text"] = df[text_col].tolist()
    return Dataset.from_dict(data).cast_column("audio", Audio(sampling_rate=16000))

train_ds = to_hf_dataset(tr_df, include_text=True)
val_ds = to_hf_dataset(va_df, include_text=True)
test_ds = to_hf_dataset(test_df, include_text=False)

processor = WhisperProcessor.from_pretrained(MODEL_NAME, task="transcribe")
processor.tokenizer.set_prefix_tokens(task="transcribe")

def prepare_train_batch(batch):
    audio = batch["audio"]
    batch["input_features"] = processor.feature_extractor(
        audio["array"],
        sampling_rate=audio["sampling_rate"]
    ).input_features[0]
    batch["labels"] = processor.tokenizer(batch["text"]).input_ids
    batch["input_length"] = len(audio["array"]) / audio["sampling_rate"]
    return batch

def prepare_test_batch(batch):
    audio = batch["audio"]
    batch["input_features"] = processor.feature_extractor(
        audio["array"],
        sampling_rate=audio["sampling_rate"]
    ).input_features[0]
    batch["input_length"] = len(audio["array"]) / audio["sampling_rate"]
    return batch

train_proc = train_ds.map(prepare_train_batch, remove_columns=train_ds.column_names)
val_proc = val_ds.map(prepare_train_batch, remove_columns=val_ds.column_names)
test_proc = test_ds.map(prepare_test_batch, remove_columns=test_ds.column_names)

sample = train_proc[0]
print("Feature shape:", np.array(sample["input_features"]).shape)
print("Example labels length:", len(sample["labels"]))


In [ ]:
@dataclass
class DataCollatorSpeechSeq2SeqWithPadding:
    processor: Any
    decoder_start_token_id: int

    def __call__(self, features: List[Dict[str, Union[List[int], np.ndarray]]]) -> Dict[str, torch.Tensor]:
        input_features = [{"input_features": f["input_features"]} for f in features]
        batch = self.processor.feature_extractor.pad(input_features, return_tensors="pt")

        if "labels" in features[0]:
            label_features = [{"input_ids": f["labels"]} for f in features]
            labels_batch = self.processor.tokenizer.pad(label_features, return_tensors="pt")

            labels = labels_batch["input_ids"].masked_fill(labels_batch.attention_mask.ne(1), -100)

            if (labels[:, 0] == self.decoder_start_token_id).all().cpu().item():
                labels = labels[:, 1:]

            batch["labels"] = labels

        return batch

data_collator = DataCollatorSpeechSeq2SeqWithPadding(
    processor=processor,
    decoder_start_token_id=processor.tokenizer.convert_tokens_to_ids("<|startoftranscript|>")
)

wer_metric = evaluate.load("wer")
cer_metric = evaluate.load("cer")

def compute_metrics(pred):
    pred_ids = pred.predictions
    if isinstance(pred_ids, tuple):
        pred_ids = pred_ids[0]

    label_ids = pred.label_ids.copy()
    label_ids[label_ids == -100] = processor.tokenizer.pad_token_id

    pred_str = processor.tokenizer.batch_decode(pred_ids, skip_special_tokens=True)
    label_str = processor.tokenizer.batch_decode(label_ids, skip_special_tokens=True)

    raw_preds = [normalize_raw_text(x) for x in pred_str]
    raw_refs = [normalize_raw_text(x) for x in label_str]
    clean_preds = [submission_cleaner(x) for x in pred_str]
    clean_refs = [submission_cleaner(x) for x in label_str]

    ref_words = np.array([max(1, len(x.split())) for x in clean_refs], dtype=np.float32)
    pred_words = np.array([len(x.split()) for x in clean_preds], dtype=np.float32)

    return {
        "raw_wer": wer_metric.compute(predictions=raw_preds, references=raw_refs),
        "raw_cer": cer_metric.compute(predictions=raw_preds, references=raw_refs),
        "cleaned_wer": wer_metric.compute(predictions=clean_preds, references=clean_refs),
        "cleaned_cer": cer_metric.compute(predictions=clean_preds, references=clean_refs),
        "exact_match_clean": float(np.mean([p == r for p, r in zip(clean_preds, clean_refs)])),
        "empty_pred_rate": float(np.mean([p == "" for p in clean_preds])),
        "pred_ref_word_ratio_clean": float(np.mean(pred_words / ref_words)),
    }

class BatchLossSeq2SeqTrainer(Seq2SeqTrainer):
    def evaluate(self, *args, **kwargs):
        self._eval_batch_losses = []
        metrics = super().evaluate(*args, **kwargs)

        if len(self._eval_batch_losses) > 0:
            arr = np.array(self._eval_batch_losses, dtype=np.float32)
            extra = {
                "eval_batch_loss_mean": float(arr.mean()),
                "eval_batch_loss_std": float(arr.std()),
                "eval_batch_loss_median": float(np.median(arr)),
                "eval_batch_loss_p90": float(np.percentile(arr, 90)),
            }
            metrics.update(extra)
            self.log(extra)

        return metrics

    def prediction_step(self, model, inputs, prediction_loss_only, ignore_keys=None):
        loss, generated_tokens, labels = super().prediction_step(
            model, inputs, prediction_loss_only, ignore_keys=ignore_keys
        )

        if hasattr(self, "_eval_batch_losses") and (loss is not None) and (not model.training):
            self._eval_batch_losses.append(loss.detach().float().cpu().item())

        return loss, generated_tokens, labels


In [ ]:
finetune_val_metrics = None
finetune_val_preds = None
hybrid_val_preds = None
hybrid_metrics = None
trainer = None

if DO_TRAIN:
    train_model = WhisperForConditionalGeneration.from_pretrained(
        MODEL_NAME,
        torch_dtype=DTYPE,
        low_cpu_mem_usage=True,
    )
    train_model.to(DEVICE)
    train_model.config.use_cache = False
    train_model.config.forced_decoder_ids = None
    train_model.generation_config.forced_decoder_ids = None
    train_model.generation_config.task = "transcribe"
    train_model.generation_config.language = None
    train_model.gradient_checkpointing_enable()

    effective_batch = PER_DEVICE_TRAIN_BATCH_SIZE * GRAD_ACCUM
    steps_per_epoch = math.ceil(len(train_proc) / effective_batch)
    eval_steps = max(10, steps_per_epoch // 2)
    logging_steps = max(5, eval_steps // 5)

    print("steps_per_epoch:", steps_per_epoch)
    print("eval_steps:", eval_steps)
    print("logging_steps:", logging_steps)

    training_args = Seq2SeqTrainingArguments(
        output_dir=OUTPUT_DIR,
        learning_rate=LEARNING_RATE,
        warmup_ratio=WARMUP_RATIO,
        weight_decay=WEIGHT_DECAY,
        per_device_train_batch_size=PER_DEVICE_TRAIN_BATCH_SIZE,
        per_device_eval_batch_size=PER_DEVICE_EVAL_BATCH_SIZE,
        gradient_accumulation_steps=GRAD_ACCUM,
        num_train_epochs=NUM_EPOCHS,
        fp16=(DEVICE == "cuda"),
        gradient_checkpointing=True,
        eval_strategy="steps",
        eval_steps=eval_steps,
        save_strategy="steps",
        save_steps=eval_steps,
        logging_steps=logging_steps,
        predict_with_generate=True,
        generation_max_length=MAX_NEW_TOKENS,
        generation_num_beams=NUM_BEAMS,
        load_best_model_at_end=True,
        metric_for_best_model=BEST_MODEL_METRIC,
        greater_is_better=False,
        save_total_limit=2,
        report_to="none",
        remove_unused_columns=False,
        label_names=["labels"],
        dataloader_num_workers=2,
        max_grad_norm=1.0,
    )

    trainer = BatchLossSeq2SeqTrainer(
        args=training_args,
        model=train_model,
        train_dataset=train_proc,
        eval_dataset=val_proc,
        data_collator=data_collator,
        compute_metrics=compute_metrics,
        processing_class=processor,
        callbacks=[EarlyStoppingCallback(early_stopping_patience=2)],
    )

    smoke_features = [train_proc[i] for i in range(min(2, len(train_proc)))]
    smoke_batch = data_collator(smoke_features)
    smoke_batch = {k: v.to(DEVICE) for k, v in smoke_batch.items()}

    train_model.eval()
    with torch.no_grad():
        smoke_out = train_model(**smoke_batch)

    print("Smoke batch keys:", list(smoke_batch.keys()))
    print("Smoke loss:", float(smoke_out.loss))
    train_model.train()

    trainer.train()
    trainer.save_model(FINAL_DIR)
    processor.save_pretrained(FINAL_DIR)

    pred_out = trainer.predict(val_proc)
    pred_ids = pred_out.predictions
    if isinstance(pred_ids, tuple):
        pred_ids = pred_ids[0]
    finetune_val_preds = processor.tokenizer.batch_decode(pred_ids, skip_special_tokens=True)
    finetune_val_metrics = compute_text_metrics(finetune_val_preds, va_df[text_col].tolist())
    print("\nFine-tuned val metrics:")
    print(pd.Series(finetune_val_metrics))

    if RUN_BASELINE_VAL and USE_BASELINE_FALLBACK:
        hybrid_val_preds, used_fallback = build_hybrid_predictions(finetune_val_preds, baseline_val_preds)
        hybrid_metrics = compute_text_metrics(hybrid_val_preds, va_df[text_col].tolist())
        print(f"\nHybrid val metrics (fallback used on {used_fallback} / {len(hybrid_val_preds)} clips):")
        print(pd.Series(hybrid_metrics))

    del smoke_batch, smoke_out
    gc.collect()
    if DEVICE == "cuda":
        torch.cuda.empty_cache()


In [ ]:
results = []

if baseline_val_metrics is not None:
    row = {"model": "baseline_zero_shot"}
    row.update(baseline_val_metrics)
    results.append(row)

if DO_TRAIN and finetune_val_metrics is not None:
    row = {"model": "full_finetune"}
    row.update(finetune_val_metrics)
    results.append(row)

if hybrid_metrics is not None:
    row = {"model": "hybrid_finetune_with_baseline_fallback"}
    row.update(hybrid_metrics)
    results.append(row)

if len(results) > 0:
    results_df = pd.DataFrame(results).sort_values(BEST_MODEL_METRIC).reset_index(drop=True)
    display(results_df)
    best_strategy = results_df.iloc[0]["model"]
else:
    best_strategy = "baseline_zero_shot"

print("Chosen strategy:", best_strategy)


In [ ]:
from torch.utils.data import DataLoader

def infer_collator(features):
    input_features = [{"input_features": f["input_features"]} for f in features]
    return processor.feature_extractor.pad(input_features, return_tensors="pt")

def predict_test_with_trainer_model(model, batch_size=PER_DEVICE_EVAL_BATCH_SIZE):
    model.eval()
    model.to(DEVICE)
    loader = DataLoader(test_proc, batch_size=batch_size, shuffle=False, collate_fn=infer_collator)
    preds = []
    for batch in tqdm(loader, desc="Fine-tuned test"):
        batch = {k: v.to(DEVICE) for k, v in batch.items()}
        with torch.no_grad():
            generated_ids = model.generate(
                input_features=batch["input_features"],
                num_beams=NUM_BEAMS,
                max_new_tokens=MAX_NEW_TOKENS,
                task="transcribe",
            )
        texts = processor.tokenizer.batch_decode(generated_ids, skip_special_tokens=True)
        preds.extend([submission_cleaner(x) for x in texts])
    return preds

if best_strategy == "baseline_zero_shot":
    final_test_preds = transcribe_paths(
        test_df["audio_path"].tolist(),
        baseline_processor,
        baseline_model,
        batch_size=PER_DEVICE_EVAL_BATCH_SIZE,
        desc="Baseline test"
    )

elif best_strategy == "full_finetune":
    final_test_preds = predict_test_with_trainer_model(trainer.model)

elif best_strategy == "hybrid_finetune_with_baseline_fallback":
    fine_test_preds = predict_test_with_trainer_model(trainer.model)
    risky_idx = [i for i, x in enumerate(fine_test_preds) if is_risky_prediction(x)]
    final_test_preds = list(fine_test_preds)
    if len(risky_idx) > 0:
        fallback_paths = test_df.iloc[risky_idx]["audio_path"].tolist()
        fallback_preds = transcribe_paths(
            fallback_paths,
            baseline_processor,
            baseline_model,
            batch_size=PER_DEVICE_EVAL_BATCH_SIZE,
            desc="Fallback baseline test"
        )
        for idx, pred in zip(risky_idx, fallback_preds):
            final_test_preds[idx] = pred
    print(f"Fallback used on {len(risky_idx)} test clips.")
else:
    raise ValueError(f"Unknown strategy: {best_strategy}")

submission = sample_sub.copy()
submission[text_col] = final_test_preds
submission.to_csv("/kaggle/working/submission.csv", index=False)
display(submission.head(10))
print("Saved:", "/kaggle/working/submission.csv")


## Optional note on the LoRA crash

Your traceback is the classic Whisper + generic PEFT wrapper mismatch:

- Whisper encoder expects `input_features`
- generic PEFT seq2seq wrappers are built around text-style `input_ids`
- the result can be errors like:
  - `unexpected keyword argument 'input_ids'`
  - `got multiple values for keyword argument 'input_ids'`

If you really want to revisit LoRA later, use one of these routes:
1. pin to a version combo known to work with Whisper examples
2. patch a Whisper subclass that swallows `input_ids` and maps it to `input_features`
3. keep the main competition notebook on the full fine-tune path above, because it is much less fragile
